# Yearly Trends in ARCHER2 Node-Hour Usage

This notebook looks at the yearly changes in workload across the top software applications used on ARCHER2.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

data_path = Path("../data/archer2_HPC_data_without_overall_and_aOut.csv")
hpc_data = pd.read_csv(data_path)

# Make sure the month column is clean and usable
hpc_data["Year"] = pd.to_numeric(hpc_data["Year"], errors="coerce")
hpc_data["Month"] = pd.to_numeric(hpc_data["Month"], errors="coerce")
hpc_data["Nodeh"] = pd.to_numeric(hpc_data["Nodeh"], errors="coerce").fillna(0)

print("Dataset loaded:", hpc_data.shape)
print(hpc_data.head())

In [ ]:
# Identify the top 5 software by total node-hours across all years
top_software = (
    hpc_data.groupby("Software")["Nodeh"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

print("Top 5 software by node-hours:")
print(top_software)

top_names = top_software.index.tolist()
filtered = hpc_data[hpc_data["Software"].isin(top_names)]

In [ ]:
# Aggregate yearly totals for each software
yearly_trend = (
    filtered.groupby(["Year", "Software"])["Nodeh"]
    .sum()
    .reset_index()
)

pivot_df = yearly_trend.pivot_table(
    index="Year",
    columns="Software",
    values="Nodeh",
    aggfunc="sum"
).fillna(0)

print(pivot_df)

In [ ]:
# Plot the yearly trend for each software
pivot_df.plot(
    kind="line",
    figsize=(12, 8),
    linewidth=2,
    marker="*",
    cmap="tab10",
    markersize=7
)

plt.title("Yearly Node-Hour Trend for the Top 5 Software (2021–2026)")
plt.xlabel("Year")
plt.ylabel("Node-hours")
plt.grid(axis="y", linestyle="-", alpha=0.6)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

# Yearly analysis and cumulative trend views

This section brings together the yearly graphing patterns from the original notebook: the overall top software, a cumulative yearly comparison, and a per-year chart for each year in the dataset.


In [ ]:
# Candidate yearly charts from the original graphing notebook
# 1) top software overall by year
# 2) cumulative yearly trend across all software
# 3) yearly trend per year with a loop

import pandas as pd
import matplotlib.pyplot as plt

hpc_data = pd.read_csv("../data/archer2_HPC_data_without_overall_and_aOut.csv")

# Clean the dataset
hpc_data["Year"] = pd.to_numeric(hpc_data["Year"], errors="coerce")
hpc_data["Month"] = pd.to_numeric(hpc_data["Month"], errors="coerce")
hpc_data["Nodeh"] = pd.to_numeric(hpc_data["Nodeh"], errors="coerce").fillna(0)

# Top 5 software overall
overall_top5 = (
    hpc_data.groupby("Software")["Nodeh"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

print("Top 5 software overall:")
print(overall_top5)

# Filter to the most used software
selected_software = overall_top5.index.tolist()
filtered = hpc_data[hpc_data["Software"].isin(selected_software)]

# Yearly Nodeh trend for top 5 software
yearly_top5 = (
    filtered.groupby(["Year", "Software"])["Nodeh"]
    .sum()
    .reset_index()
)

pivot_df = yearly_top5.pivot_table(
    index="Year",
    columns="Software",
    values="Nodeh",
    aggfunc="sum"
).fillna(0)

pivot_df.plot(
    kind="line",
    figsize=(12, 8),
    linewidth=2,
    marker="*",
    cmap="tab10",
    markersize=7
)

plt.title("Yearly Node-Hour Trend for Top 5 Software")
plt.xlabel("Year")
plt.ylabel("Node-hours")
plt.grid(axis="y", linestyle="-", alpha=0.6)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

# Cumulative yearly trend for all software
all_yearly = (
    hpc_data.groupby("Year")["Nodeh"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(12, 7))
plt.plot(all_yearly["Year"], all_yearly["Nodeh"], marker="*", linewidth=2, color="darkblue")
plt.title("Cumulative Yearly Node-Hour Trend")
plt.xlabel("Year")
plt.ylabel("Node-hours")
plt.grid(True)
plt.tight_layout()
plt.show()

# Per-year trend plots for each year
for year_name, year_group in filtered.groupby("Year"):
    monthly_trend = (
        year_group.groupby(["Month", "Software"])["Nodeh"]
        .sum()
        .reset_index()
    )

    pivot_year = monthly_trend.pivot_table(
        index="Month",
        columns="Software",
        values="Nodeh",
        aggfunc="sum"
    ).fillna(0)

    fig, ax = plt.subplots(figsize=(12, 7))
    pivot_year.plot(kind="line", ax=ax, marker="*", linewidth=2)
    ax.set_title(f"Monthly Node-Hour Trend for Top 5 Software in {int(year_name)}")
    ax.set_xlabel("Month")
    ax.set_ylabel("Node-hours")
    ax.grid(True)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()
